In [ ]:
import torch
import torch.nn as nn

# A single logistic unit with 2 inputs
# w = [1, 1], b = -3  →  decision boundary: x1 + x2 = 3 <=> x2 = -x1 + 3
unit = nn.Linear(2, 1)                    # wraps wᵀx + b
unit.weight.data = torch.tensor([[1., 1.]])
unit.bias.data   = torch.tensor([-3.])

# Test a few points
points = torch.tensor([
    [1., 1.],   # x1+x2=2 < 3  → should output < 0.5 (class 0)
    [2., 2.],   # x1+x2=4 > 3  → should output > 0.5 (class 1)
    [1.5, 1.5], # x1+x2=3 = 3  → exactly on the boundary → 0.5
])

logits = unit(points)                     # raw wᵀx + b
probs  = torch.sigmoid(logits)            # apply σ
for pt, pr in zip(points, probs):
    cls = 1 if pr > 0.5 else 0
    print(f'x={pt.tolist()}  P(y=1)={pr.item():.3f}  → class {cls}')

# Output:
# x=[1.0, 1.0]  P(y=1)=0.269  → class 0
# x=[2.0, 2.0]  P(y=1)=0.731  → class 1
# x=[1.5, 1.5]  P(y=1)=0.500  → class 0  (tie goes to 0)

x=[1.0, 1.0]  P(y=1)=0.269  → class 0
x=[2.0, 2.0]  P(y=1)=0.731  → class 1
x=[1.5, 1.5]  P(y=1)=0.500  → class 0


In [2]:
import torch
import torch.nn as nn

# ── Building an MLP step by step ─────────────────────────────────────

# Architecture: input(3) → hidden1(4) → hidden2(4) → output(2)
# This is a 3-layer network (L=3, not counting input as a layer)

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        # Each nn.Linear wraps W⁽ˡ⁾ and b⁽ˡ⁾
        self.layer1 = nn.Linear(input_dim,  hidden_dim)   # W: (hidden × input)
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)   # W: (hidden × hidden)
        self.layer3 = nn.Linear(hidden_dim, output_dim)   # W: (output × hidden)

    def forward(self, x):
        # Forward propagation: z = Wa + b,  a = σ(z)
        a1 = torch.sigmoid(self.layer1(x))   # hidden layer 1
        a2 = torch.sigmoid(self.layer2(a1))  # hidden layer 2
        a3 = self.layer3(a2)                 # output logits (no activation yet)
        return a3

model = MLP(input_dim=3, hidden_dim=4, output_dim=2)

# Inspect weight shapes
for name, param in model.named_parameters():
    print(f'{name:20s}  shape: {tuple(param.shape)}')

# layer1.weight       shape: (4, 3)   ← W⁽¹⁾ ∈ ℝ^{4×3}
# layer1.bias         shape: (4,)     ← b⁽¹⁾ ∈ ℝ^4
# layer2.weight       shape: (4, 4)   ← W⁽²⁾ ∈ ℝ^{4×4}
# layer2.bias         shape: (4,)     ← b⁽²⁾ ∈ ℝ^4
# layer3.weight       shape: (2, 4)   ← W⁽³⁾ ∈ ℝ^{2×4}
# layer3.bias         shape: (2,)     ← b⁽³⁾ ∈ ℝ^2

# Total parameters
total = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total}')   # 4*3+4 + 4*4+4 + 2*4+2 = 46


layer1.weight         shape: (4, 3)
layer1.bias           shape: (4,)
layer2.weight         shape: (4, 4)
layer2.bias           shape: (4,)
layer3.weight         shape: (2, 4)
layer3.bias           shape: (2,)
Total parameters: 46


In [ ]:
import torch
import torch.nn as nn

# ── Manual forward propagation vs nn.Module ───────────────────────────
torch.manual_seed(0)

# Define a tiny 2-layer network: 3 → 4 → 2
W1 = torch.randn(4, 3, requires_grad=True)
b1 = torch.zeros(4,    requires_grad=True)
W2 = torch.randn(2, 4, requires_grad=True)
b2 = torch.zeros(2,    requires_grad=True)

x = torch.randn(3)   # one input vector

# Manual forward pass — shows every intermediate value
z1 = W1 @ x + b1                  # pre-activation layer 1,  shape (4,)
a1 = torch.sigmoid(z1)            # activation layer 1,      shape (4,)
z2 = W2 @ a1 + b2                 # pre-activation layer 2,  shape (2,)
a2 = torch.sigmoid(z2)            # output probabilities,    shape (2,)

print('z1:', z1.detach().round(decimals=3))
print('a1:', a1.detach().round(decimals=3))  # values in (0,1)
print('z2:', z2.detach().round(decimals=3))
print('a2:', a2.detach().round(decimals=3))  # values in (0,1)

# All intermediate tensors (z1, a1, z2, a2) are retained in the
# computation graph — PyTorch will use them during backward().

# ── The same with nn.Sequential (cleaner, same computation) ──────────
net = nn.Sequential(
    nn.Linear(3, 4),
    nn.Sigmoid(),
    nn.Linear(4, 2),
    nn.Sigmoid(),
)
out = net(x)
print('nn.Sequential output:', out.detach().round(decimals=3))


z1: tensor([-1.9040, -2.6830,  0.6940, -0.8470])
a1: tensor([0.1300, 0.0640, 0.6670, 0.3000])
z2: tensor([-0.7180, -1.0970])
a2: tensor([0.3280, 0.2500])
nn.Sequential output: tensor([0.5900, 0.6790])


In [4]:
import torch
import torch.nn as nn

# ── Multi-class classification (mutually exclusive classes) ───────────
# nn.CrossEntropyLoss = log_softmax + NLLLoss
# IMPORTANT: expects raw LOGITS, not probabilities
ce_loss = nn.CrossEntropyLoss()

logits = torch.tensor([[2.0, 0.5, -1.0]])  # 1 example, 3 classes
target = torch.tensor([0])                  # true class is 0

loss = ce_loss(logits, target)
print(f'Cross-entropy loss: {loss:.4f}')

# Manual calculation to verify:
probs    = torch.softmax(logits, dim=1)
manual   = -torch.log(probs[0, 0])          # -log P(class 0)
print(f'Manual:             {manual:.4f}')   # should match

# ── Multi-label classification (non-exclusive classes) ────────────────
# nn.BCEWithLogitsLoss = sigmoid + binary CE, applied element-wise
bce_loss = nn.BCEWithLogitsLoss()

logits_ml = torch.tensor([[1.5, -0.5, 2.0]])          # 3 independent outputs
target_ml = torch.tensor([[1.0,  0.0, 1.0]])           # classes 0 and 2 are present
print(f'Multi-label BCE: {bce_loss(logits_ml, target_ml):.4f}')

# ── Quadratic (L2) loss — less common but useful to know ─────────────
mse = nn.MSELoss()
pred   = torch.tensor([[0.8, 0.1, 0.1]])
target_oh = torch.tensor([[1.0, 0.0, 0.0]])
print(f'MSE loss: {mse(pred, target_oh):.4f}')


Cross-entropy loss: 0.2413
Manual:             0.2413
Multi-label BCE: 0.2675
MSE loss: 0.0200


In [5]:
import torch

# ── Manual backpropagation for a 2-layer network ──────────────────────
# Architecture: input(2) → hidden(3) → output(1)
# Loss: MSE (quadratic)

torch.manual_seed(42)
W1 = torch.randn(3, 2)    # W⁽¹⁾ ∈ ℝ^{3×2}
b1 = torch.zeros(3)       # b⁽¹⁾ ∈ ℝ^3
W2 = torch.randn(1, 3)    # W⁽²⁾ ∈ ℝ^{1×3}
b2 = torch.zeros(1)       # b⁽²⁾ ∈ ℝ^1

# One training example
x = torch.tensor([0.5, -0.3])
y = torch.tensor([1.0])

def sigmoid(z):    return 1 / (1 + torch.exp(-z))
def sigmoid_d(z):  return sigmoid(z) * (1 - sigmoid(z))  # σ'(z)

# ── FORWARD PASS ──────────────────────────────────────────────────────
z1 = W1 @ x + b1              # pre-activation layer 1,  (3,)
a1 = sigmoid(z1)              # activation layer 1,      (3,)
z2 = W2 @ a1 + b2             # pre-activation output,   (1,)
a2 = sigmoid(z2)              # network output ŷ,        (1,)

loss = 0.5 * ((a2 - y) ** 2).sum()   # MSE loss
print(f'Forward: ŷ={a2.item():.4f}, y={y.item()}, loss={loss.item():.4f}')

# ── BACKWARD PASS ─────────────────────────────────────────────────────
# Equation 1: error at output layer
delta2 = (a2 - y) * sigmoid_d(z2)    # δ⁽ᴸ⁾ = (ŷ - y) ⊙ σ'(z⁽ᴸ⁾)

# Equation 3 & 4: gradients for W2 and b2
dW2 = delta2.unsqueeze(1) * a1.unsqueeze(0)   # ∂J/∂W⁽²⁾ = δ⁽³⁾ aᵀ⁽²⁾
db2 = delta2                                    # ∂J/∂b⁽²⁾ = δ⁽³⁾

# Equation 2: backpropagate error to layer 1
delta1 = (W2.T @ delta2) * sigmoid_d(z1)       # δ⁽²⁾ = W⁽²⁾ᵀ δ⁽³⁾ ⊙ σ'(z⁽²⁾)

# Equation 3 & 4: gradients for W1 and b1
dW1 = delta1.unsqueeze(1) * x.unsqueeze(0)     # ∂J/∂W⁽¹⁾ = δ⁽²⁾ aᵀ⁽¹⁾
db1 = delta1                                    # ∂J/∂b⁽¹⁾ = δ⁽²⁾

print(f'dW2: {dW2}')
print(f'dW1: {dW1}')

# ── VERIFY WITH PYTORCH AUTOGRAD ──────────────────────────────────────
W1_t = W1.clone().requires_grad_(True)
b1_t = b1.clone().requires_grad_(True)
W2_t = W2.clone().requires_grad_(True)
b2_t = b2.clone().requires_grad_(True)

a1_t = sigmoid(W1_t @ x + b1_t)
a2_t = sigmoid(W2_t @ a1_t + b2_t)
loss_t = 0.5 * ((a2_t - y) ** 2).sum()
loss_t.backward()

print('\nManual dW2 matches autograd:', torch.allclose(dW2, W2_t.grad, atol=1e-6))
print('Manual dW1 matches autograd:', torch.allclose(dW1, W1_t.grad, atol=1e-6))
# Both should print True


Forward: ŷ=0.7355, y=1.0, loss=0.0350
dW2: tensor([[-0.0274, -0.0263, -0.0194]])
dW1: tensor([[-0.0141,  0.0085],
        [ 0.0041, -0.0025],
        [-0.0028,  0.0017]])

Manual dW2 matches autograd: True
Manual dW1 matches autograd: True


In [18]:
import torch
import torch.nn as nn

# ── Demonstrating gradient flow with sigmoid vs ReLU ──────────────────
torch.manual_seed(0)

def build_network(activation, depth=10):
    layers = []
    for _ in range(depth):
        layers += [nn.Linear(64, 64), activation()]
    layers.append(nn.Linear(64, 1))
    return nn.Sequential(*layers)

x      = torch.randn(1, 64)
target = torch.ones(1, 1)

for act_name, act_cls in [('Sigmoid', nn.Sigmoid), ('ReLU', nn.ReLU)]:
    net    = build_network(act_cls, depth=10)
    loss   = nn.MSELoss()(net(x), target)
    loss.backward()

    # Look at gradient magnitude in the FIRST layer
    first_grad = net[0].weight.grad.abs().mean().item()
    print(f'{act_name:s}  first-layer gradient mean: {first_grad:.12f}')

# Sigmoid → gradient is ~0.0000000001 (vanished over 10 layers)
# ReLU    → gradient is ~0.00001      (healthier, doesn't vanish completely)


Sigmoid  first-layer gradient mean: 0.000000000356
ReLU  first-layer gradient mean: 0.000014990145


In [19]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split

# ── Training loop with loss curve monitoring ──────────────────────────
torch.manual_seed(0)
N = 1000
X = torch.randn(N, 20)
y = (X[:, 0] + X[:, 1] > 0).long()       # simple rule
dataset    = TensorDataset(X, y)
train_ds, val_ds = random_split(dataset, [800, 200])

model     = nn.Sequential(nn.Linear(20,64), nn.ReLU(),
                           nn.Linear(64,64), nn.ReLU(),
                           nn.Linear(64, 2))
optimiser = torch.optim.SGD(model.parameters(), lr=0.01, weight_decay=1e-3)
loss_fn   = nn.CrossEntropyLoss()

train_losses, val_losses = [], []
best_val_loss, best_epoch = float('inf'), 0

for epoch in range(100):
    # ── Training ──────────────────────────────────────────────────────
    model.train()
    epoch_loss = 0
    for X_b, y_b in DataLoader(train_ds, batch_size=64, shuffle=True):
        optimiser.zero_grad()
        loss = loss_fn(model(X_b), y_b)
        loss.backward()
        optimiser.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_ds) * 64)

    # ── Validation ────────────────────────────────────────────────────
    model.eval()
    with torch.no_grad():
        X_val, y_val = val_ds[:]
        val_loss = loss_fn(model(X_val), y_val).item()
    val_losses.append(val_loss)

    # Early stopping: save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch    = epoch
        best_state    = {k: v.clone() for k, v in model.state_dict().items()}

print(f'Best epoch: {best_epoch}  (val loss: {best_val_loss:.4f})')

# Restore best weights before evaluation on test set
model.load_state_dict(best_state)

Best epoch: 99  (val loss: 0.1475)


<All keys matched successfully>

In [20]:
import torch

# ── Computational graph example: f(x,y,z) = (x + y) * z ──────────────
x = torch.tensor(-2.0, requires_grad=True)
y = torch.tensor( 5.0, requires_grad=True)
z = torch.tensor( 4.0, requires_grad=True)

# Forward pass — PyTorch builds the graph as we compute
q = x + y        # add gate
f = q * z        # multiply gate

print(f'Forward: q={q.item()}, f={f.item()}')  # q=3, f=12

# Backward pass — one call propagates all gradients
f.backward()

# Chain rule results:
# ∂f/∂x = z = 4  (gradient flows through q to x via add gate)
# ∂f/∂y = z = 4  (same path)
# ∂f/∂z = q = 3  (direct from multiply gate)
print(f'∂f/∂x = {x.grad.item()}')   # 4.0
print(f'∂f/∂y = {y.grad.item()}')   # 4.0
print(f'∂f/∂z = {z.grad.item()}')   # 3.0

# ── Gradient check: verify with finite differences ────────────────────
eps = 1e-5
for var, name in [(x, 'x'), (y, 'y'), (z, 'z')]:
    with torch.no_grad():
        var.data += eps
        f_plus = (x + y) * z
        var.data -= 2 * eps
        f_minus = (x + y) * z
        var.data += eps          # restore
        fd_grad = (f_plus - f_minus) / (2 * eps)
    print(f'Finite diff ∂f/∂{name} ≈ {fd_grad.item():.4f}  (autograd: {var.grad.item():.4f})')

Forward: q=3.0, f=12.0
∂f/∂x = 4.0
∂f/∂y = 4.0
∂f/∂z = 3.0
Finite diff ∂f/∂x ≈ 4.0054  (autograd: 4.0000)
Finite diff ∂f/∂y ≈ 4.0054  (autograd: 4.0000)
Finite diff ∂f/∂z ≈ 3.0518  (autograd: 3.0000)


In [21]:
import torch
import torch.nn as nn

# ── PyTorch autograd on a full MLP — nothing manual needed ────────────
torch.manual_seed(0)

model = nn.Sequential(
    nn.Linear(4, 8), nn.Sigmoid(),
    nn.Linear(8, 8), nn.Sigmoid(),
    nn.Linear(8, 3),
)

x      = torch.randn(16, 4)   # mini-batch of 16 examples
target = torch.randint(0, 3, (16,))

# Forward pass builds the computation graph
logits = model(x)
loss   = nn.CrossEntropyLoss()(logits, target)

# Backward pass — ONE call computes ALL gradients
loss.backward()

# Every parameter now has its gradient populated
for name, param in model.named_parameters():
    print(f'{name:20s}  grad norm: {param.grad.norm():.4f}')

# ── What autograd tracks ──────────────────────────────────────────────
# Every tensor created from a requires_grad=True tensor
# records the operation that created it. This forms the graph.
# backward() traverses the graph in reverse, applying the chain rule
# at each node using the stored forward-pass values.

# You can inspect the graph:
x_single = torch.randn(2, requires_grad=True)
z = (x_single ** 2).sum()
print(z.grad_fn)                    # shows the operation: SumBackward
print(z.grad_fn.next_functions)     # shows previous ops in the graph

0.weight              grad norm: 0.0050
0.bias                grad norm: 0.0029
2.weight              grad norm: 0.0283
2.bias                grad norm: 0.0195
4.weight              grad norm: 0.2504
4.bias                grad norm: 0.1805
((<PowBackward0 object at 0x7d444c2fe5f0>, 0),)


In [22]:
import torch
import torch.nn as nn

# ── Autograd through a convolutional layer ────────────────────────────
# PyTorch handles backprop for conv exactly as described above.
# We can verify by checking gradient shapes.

conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=2, bias=False)

# Input: 1 image, 1 channel, 3×3
X = torch.randn(1, 1, 3, 3, requires_grad=True)
O = conv(X)                      # output: (1, 1, 2, 2)
loss = O.sum()                   # simple scalar loss
loss.backward()

print('Input grad shape:  ', X.grad.shape)           # (1, 1, 3, 3) = same as X
print('Filter grad shape: ', conv.weight.grad.shape) # (1, 1, 2, 2) = same as F

# ∂L/∂F = conv(X, ∂L/∂O)  — verify manually
# ∂L/∂O = all ones (since loss = O.sum())
dL_dO = torch.ones_like(O)

# Manual ∂L/∂F: for a sum loss, this is just the sum of all input patches
# covered by each filter position — which equals a 'valid' conv of X with dL/dO
manual_dF = nn.functional.conv2d(X.detach(), dL_dO, padding=0)
print('Manual dF matches autograd:', torch.allclose(manual_dF, conv.weight.grad, atol=1e-5))

Input grad shape:   torch.Size([1, 1, 3, 3])
Filter grad shape:  torch.Size([1, 1, 2, 2])
Manual dF matches autograd: True
